# CSCI1409 BONUS — Traffic Sign Detection with YOLO11n

**Individual object detection project · 4 classes · two controlled runs**

**What this notebook actually does:** downloads public street-scene images with PASCAL VOC boxes (877 images described by the source), audits all labels, creates separate train/validation/test sets without cross-split perceptual duplicates, converts VOC to YOLO, shows ≥5 ground-truth examples, fine-tunes the **same pretrained YOLO11n** twice, changes **only brightness augmentation** (`hsv_v=0.0` vs `hsv_v=0.35`), compares validation Precision/Recall/mAP@0.50/time, selects by validation mAP50, then tests once and shows ≥5 true predictions and ≥3 checked failure candidates.

**Source and attribution:** [Kaggle Road Sign Detection by andrewmvd](https://www.kaggle.com/datasets/andrewmvd/road-sign-detection), redistributed as a prepared VOC archive by [PaddleDetection](https://github.com/PaddlePaddle/PaddleDetection/blob/release/2.9/docs/tutorials/data/PrepareDetDataSet_en.md). Image rights/license: **verify original Kaggle terms before redistribution**; we keep the original images out of GitHub. YOLO/Ultralytics code uses AGPL-3.0 unless covered by another license.

**Honesty rule:** metrics, prediction examples and trained `.pt` weights in this project are *not pre-filled*. They are produced by running this notebook. A GPU (Runtime → Change runtime type → T4 GPU) is recommended; paid Colab is not required. Run **Runtime → Run all**.

## 0 · Install dependencies and set reproducible environment
Ultralytics version is pinned. Colab's own CUDA-enabled `torch` installation is used, so we avoid reinstalling PyTorch. All random seeds are recorded. Full training requires a network connection for dataset and pretrained weights.

In [ ]:
import sys, subprocess
from importlib.metadata import version, PackageNotFoundError
packages={'ultralytics':'8.3.220', 'ImageHash':'4.3.2', 'PyYAML':'6.0.2',
          'python-pptx':'1.0.2', 'pandas':'2.2.3'}
required=[]
for distribution,expected in packages.items():
    try: actual=version(distribution)
    except PackageNotFoundError: actual=None
    if actual!=expected: required.append(f'{distribution}=={expected}')
if required:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*required])

import csv, json, time, random, hashlib, shutil, tarfile, urllib.request, platform, math, os
from pathlib import Path
from collections import Counter, defaultdict
import xml.etree.ElementTree as ET
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
import imagehash, yaml, torch, ultralytics
from ultralytics import YOLO
from IPython.display import display

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=0 if torch.cuda.is_available() else 'cpu'
BASE=Path('/content/traffic_sign_bonus') if Path('/content').exists() else Path.cwd()/'traffic_sign_bonus'
SRC=BASE/'source'; DATA=BASE/'dataset'; RUNS=BASE/'runs'; EVIDENCE=BASE/'evidence'; WEIGHTS=BASE/'weights'
for d in [SRC,DATA,RUNS,EVIDENCE,WEIGHTS]:d.mkdir(parents=True,exist_ok=True)
CLASSES=['crosswalk','speedlimit','stop','trafficlight']
ENV={'python':sys.version.split()[0],'ultralytics':ultralytics.__version__,
     'torch':torch.__version__,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
     'seed':SEED,'device':str(DEVICE)}
(EVIDENCE/'environment.json').write_text(json.dumps(ENV,indent=2))
print(ENV)

## 1 · Download Road Sign Detection (VOC) and create leakage-resistant train/val/test
The documented collection has **877 images and four classes**. The PaddleDetection archive supplies `train.txt` and `valid.txt` for a 701/176 partition. We reserve the provider `valid.txt` images as our **locked final test**, split the training pool deterministically with `seed=42`, and group exact/near-duplicate images by perceptual hash. If a near-duplicate connects to the locked test, the entire group stays in test. We do not have original video IDs; we cannot independently verify video-level grouping.

**Important:** Numeric split sizes and per-class object counts below are computed from actual files, not assumed.

In [ ]:
DATA_URL='https://paddlemodels.bj.bcebos.com/object_detection/roadsign_voc.tar'
ARCHIVE=SRC/'roadsign_voc.tar'
def find_voc_root():
    for p in [SRC,*SRC.rglob('*')]:
        if p.is_dir() and (p/'images').is_dir() and (p/'annotations').is_dir():return p
    return None
VOC_ROOT=find_voc_root()
if VOC_ROOT is None:
    print('Downloading the VOC road-sign archive from PaddleDetection ...')
    urllib.request.urlretrieve(DATA_URL, ARCHIVE)
    with tarfile.open(ARCHIVE,'r:*') as tf:
        root=SRC.resolve()
        for member in tf.getmembers():
            target=(root/member.name).resolve()
            if not target.is_relative_to(root) or member.issym() or member.islnk():
                raise ValueError(f'Unsafe tar entry: {member.name}')
        tf.extractall(SRC,filter='data') if sys.version_info >= (3,12) else tf.extractall(SRC)
    VOC_ROOT=find_voc_root()
if VOC_ROOT is None: raise FileNotFoundError('VOC images/ and annotations/ folders not found in archive')
print('VOC data directory:',VOC_ROOT)

IMG_EXT={'.jpg','.jpeg','.png','.webp'}
all_images=sorted(p for p in (VOC_ROOT/'images').iterdir() if p.suffix.lower() in IMG_EXT)
assert len(all_images)>=300,f'Too few source images: {len(all_images)}'
source_lookup={p.stem:p for p in all_images}
assert len(source_lookup)==len(all_images),'Repeated image stems; cannot match XML safely'
name_alias={'trafficsignal':'trafficlight','traffic light':'trafficlight','speed limit':'speedlimit',
            'pedestrian crossing':'crosswalk'}
def parse_voc(path):
    xml=VOC_ROOT/'annotations'/f'{path.stem}.xml'
    if not xml.exists(): raise FileNotFoundError('Missing XML label: '+str(xml))
    tree=ET.parse(xml).getroot()
    with Image.open(path) as img:width,height=img.size
    objects=[]
    for obj in tree.findall('object'):
        raw=(obj.findtext('name') or '').strip().lower()
        name=name_alias.get(raw,raw.replace(' ','').replace('_',''))
        if name not in CLASSES: raise ValueError(f'Unknown category {raw!r} in {xml}')
        box=obj.find('bndbox')
        if box is None:raise ValueError(f'Missing bounding box in {xml}')
        xmin=float(box.findtext('xmin'));ymin=float(box.findtext('ymin'))
        xmax=float(box.findtext('xmax'));ymax=float(box.findtext('ymax'))
        if not (0<=xmin<xmax<=width+1 and 0<=ymin<ymax<=height+1):
            raise ValueError(f'Invalid bounding box in {xml}: {(xmin,ymin,xmax,ymax)} for {(width,height)}')
        xmin=max(0,xmin);ymin=max(0,ymin);xmax=min(width,xmax);ymax=min(height,ymax)
        xc=(xmin+xmax)/(2*width);yc=(ymin+ymax)/(2*height)
        w=(xmax-xmin)/width;h=(ymax-ymin)/height
        if w<=0 or h<=0:raise ValueError(f'Empty bounding box in {xml}')
        objects.append((CLASSES.index(name),xc,yc,w,h))
    return objects
annotations={p.stem:parse_voc(p) for p in all_images}
print('Source images:',len(all_images),'| Labels:',sum(map(len,annotations.values())))

def read_source_stems(filename):
    f=VOC_ROOT/filename
    if not f.exists():return set()
    stems=set()
    for line in f.read_text(errors='replace').splitlines():
        if not line.strip():continue
        tokens=line.split()
        candidates=[Path(x).stem for x in tokens if Path(x).suffix.lower() in IMG_EXT]
        if candidates:stems.add(candidates[0])
    return stems
source_test=read_source_stems('test.txt') or read_source_stems('valid.txt') or read_source_stems('val.txt')
if source_test:
    source_test=source_test.intersection(set(source_lookup))
    print('Provider hold-out images:',len(source_test))
else:
    print('Source split lists not found: will make seeded grouped 70/15/15 split')

# Inspect ALL images and group exact + near duplicates before splitting.
checksums={p.stem:hashlib.sha256(p.read_bytes()).hexdigest() for p in all_images}
phashes={p.stem:imagehash.phash(Image.open(p).convert('RGB')) for p in all_images}
stems=sorted(source_lookup)
parent={k:k for k in stems}
def root(k):
    while parent[k]!=k:
        parent[k]=parent[parent[k]];k=parent[k]
    return k
def union(a,b):
    ra,rb=root(a),root(b)
    if ra!=rb:parent[rb]=ra
# pHash Hamming distance <=4 is a conservative visual near-duplicate screen.
for i,a in enumerate(stems):
    for b in stems[i+1:]:
        if checksums[a]==checksums[b] or (phashes[a]-phashes[b])<=4: union(a,b)
groups=defaultdict(list)
for s in stems:groups[root(s)].append(s)
image_groups=sorted(groups.values(),key=lambda g:g[0])

rng=random.Random(SEED)
if source_test:
    test_groups=[g for g in image_groups if any(x in source_test for x in g)]
    pool_groups=[g for g in image_groups if not any(x in source_test for x in g)]
    rng.shuffle(pool_groups)
    val_groups=[];train_groups=[];target=sum(map(len,pool_groups))*0.20;acc=0
    for g in pool_groups:
        if acc<target:val_groups.append(g);acc+=len(g)
        else:train_groups.append(g)
else:
    shuffled=image_groups[:];rng.shuffle(shuffled)
    n=sum(map(len,shuffled));test_groups=[];val_groups=[];train_groups=[]
    t=v=0
    for g in shuffled:
        if t<n*0.15:test_groups.append(g);t+=len(g)
        elif v<n*0.15:val_groups.append(g);v+=len(g)
        else:train_groups.append(g)
assigned={s:split for split,gs in [('train',train_groups),('val',val_groups),('test',test_groups)]
          for g in gs for s in g}
assert set(assigned)==set(stems) and len(assigned)==len(stems)
# Rebuild processed split folders from scratch when this cell is rerun.
for sub in ('images','labels'):
    if (DATA/sub).exists():shutil.rmtree(DATA/sub)
for split in ('train','val','test'):
    (DATA/'images'/split).mkdir(parents=True,exist_ok=True)
    (DATA/'labels'/split).mkdir(parents=True,exist_ok=True)

for stem,split in assigned.items():
    p=source_lookup[stem]
    dst=DATA/'images'/split/p.name
    if not dst.exists():shutil.copy2(p,dst)
    rows=annotations[stem]
    (DATA/'labels'/split/f'{stem}.txt').write_text('\n'.join(
        f'{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}' for c,x,y,w,h in rows)+'\n',encoding='utf-8')
DATA_YAML=BASE/'traffic_signs.yaml'
DATA_YAML.write_text(yaml.safe_dump({'path':str(DATA.resolve()),'train':'images/train',
              'val':'images/val','test':'images/test','names':{i:n for i,n in enumerate(CLASSES)}},sort_keys=False))
manifest=pd.DataFrame([{'image':s,'split':assigned[s],'source_holdout':s in source_test,
           'sha256':checksums[s],'phash':str(phashes[s]),'group_id':root(s)} for s in stems])
manifest.to_csv(EVIDENCE/'split_manifest.csv',index=False)
split_files={k:sorted((DATA/'images'/k).glob('*')) for k in ('train','val','test')}
assert all(split_files.values()),'A split has no images'
counts=[]
for k,paths in split_files.items():
    freq=Counter()
    for p in paths:freq.update(c for c,*_ in annotations[p.stem])
    counts.append({'split':k,'images':len(paths),'objects':sum(freq.values()),
                   **{n:freq[i] for i,n in enumerate(CLASSES)}})
summary=pd.DataFrame(counts)
summary.to_csv(EVIDENCE/'dataset_counts.csv',index=False)
(EVIDENCE/'dataset_audit.json').write_text(json.dumps({
    'source':DATA_URL,'source_license':'Check the original Kaggle source terms',
    'seed':SEED,'split_method':'Source held-out subset + seeded grouped train/val, phash<=4',
    'provider_test_images':len(source_test),'perceptual_groups':len(image_groups),
    'image_count':len(all_images),'object_count':sum(map(len,annotations.values())),
    'group_check':'No exact/perceptual near-duplicate group crosses splits',
    'video_id_warning':'Source video IDs unavailable; video-level grouping unverified',
    'counts':counts},indent=2))
assert manifest.groupby('group_id')['split'].nunique().max()==1
print('Leakage-resistant grouped split, seed=42:');display(summary)
print('Dataset YAML:',DATA_YAML)

## 2 · Display five real ground-truth examples
Bounding boxes in YOLO format are `class_id x_center y_center width height` with coordinates normalized to [0,1]. We check XML files and draw real annotations. **Do not confuse these images with model predictions.**

In [ ]:
def draw_ground_truth(p):
    image=Image.open(p).convert('RGB');w,h=image.size
    d=ImageDraw.Draw(image)
    for c,xc,yc,bw,bh in annotations[p.stem]:
        box=((xc-bw/2)*w,(yc-bh/2)*h,(xc+bw/2)*w,(yc+bh/2)*h)
        d.rectangle(box,outline='#22E8B0',width=max(2,w//220))
        d.text((box[0],max(0,box[1]-16)),CLASSES[c],fill='#22E8B0',stroke_width=1,stroke_fill='black')
    return image
show=[]
for c in range(len(CLASSES)):
    match=next((p for p in split_files['train'] if c in [r[0] for r in annotations[p.stem]] and p not in show),None)
    if match:show.append(match)
for p in split_files['train']:
    if len(show)>=6:break
    if p not in show:show.append(p)
assert len(show)>=5
fig,axes=plt.subplots(2,3,figsize=(15,9))
for ax,p in zip(axes.ravel(),show[:6]):
    ax.imshow(draw_ground_truth(p));ax.set_title(p.stem);ax.axis('off')
plt.suptitle('Real GROUND-TRUTH boxes: four road sign categories',fontsize=15)
plt.tight_layout();plt.savefig(EVIDENCE/'ground_truth_examples.png',dpi=150,bbox_inches='tight');plt.show()

## 3 · Two controlled experiments: brightness augmentation

**Hypothesis (before training):** A moderate HSV brightness jitter (`hsv_v=0.35`) may increase generalization to darker/brighter road scenes compared with no brightness jitter. Only `hsv_v` changes. Both runs begin with **identical COCO-pretrained `yolo11n.pt` weights**, identical train/val/test split, epochs, image size, batch size, learning rate, seed, optimizer and every other augmentation parameter. **An improvement is not required**; we report results honestly.

In [ ]:
COMMON={'model':'yolo11n.pt','imgsz':512,'epochs':14,'batch':16,'lr0':0.01,
        'optimizer':'SGD','seed':SEED,'deterministic':True,'patience':100,'workers':2,
        'mosaic':0.0,'mixup':0.0,'degrees':0.0,'translate':0.0,'scale':0.0,
        'shear':0.0,'perspective':0.0,'flipud':0.0,'fliplr':0.0,
        'hsv_h':0.0,'hsv_s':0.0,'close_mosaic':0,'weight_decay':0.0005,'cos_lr':False}
(EVIDENCE/'controlled_experiment_config.json').write_text(json.dumps({**COMMON,
    'baseline_hsv_v':0.0,'experiment_hsv_v':0.35,'training_data':str(DATA_YAML),
    'hypothesis':'Brightness jitter may improve grouped validation mAP@0.50',
    'model_selection':'validation mAP@0.50 only'},indent=2))
print('Fixed settings:');print(json.dumps(COMMON,indent=2))

In [ ]:
times_path=EVIDENCE/'training_times.json'
train_times=json.loads(times_path.read_text()) if times_path.exists() else {}
def train_one(name,hsv_v):
    dst=RUNS/name
    best=dst/'weights'/'best.pt'
    # Always train from identical original pretrained weights on a fresh run.
    model=YOLO(COMMON['model'])  # SAME pretrained initialization for both runs
    params={k:v for k,v in COMMON.items() if k!='model'}
    t0=time.perf_counter()
    model.train(data=str(DATA_YAML),device=DEVICE,project=str(RUNS),name=name,
                exist_ok=True,plots=True,save=True,hsv_v=hsv_v,
                amp=torch.cuda.is_available(),**params)
    train_times[name]=round(time.perf_counter()-t0,2)
    times_path.write_text(json.dumps(train_times,indent=2))
    if not best.exists():raise FileNotFoundError(f'Missing best.pt for {name}')
    return best
baseline_best=train_one('baseline_no_brightness',0.0)
experiment_best=train_one('experiment_brightness_035',0.35)
print('Train times (seconds):',train_times)

## 4 · Compare BOTH checkpoints on validation ONLY
Metrics: **Precision**, **Recall**, **mAP@0.50**; the test remains locked. Evaluation confidence cutoff is `0.001` and prediction/NMS IoU threshold is `0.7`; AP matching IoU for mAP@0.50 is `0.50` and is a *different* parameter.

In [ ]:
EVAL_CONF=0.001
EVAL_NMS_IOU=0.7
def evaluate(path,split,label):
    scores=YOLO(str(path)).val(data=str(DATA_YAML),split=split,imgsz=COMMON['imgsz'],
                   batch=COMMON['batch'],conf=EVAL_CONF,iou=EVAL_NMS_IOU,
                   device=DEVICE,plots=True,verbose=False,
                   project=str(BASE/'evaluations'),name=f'{label}_{split}',exist_ok=True)
    return {'precision':float(scores.box.mp),'recall':float(scores.box.mr),
            'mAP50':float(scores.box.map50),'mAP50_95':float(scores.box.map),
            'eval_conf':EVAL_CONF,'nms_iou':EVAL_NMS_IOU,'ap_match_iou':0.5,
            'split':split}
val_file=EVIDENCE/'validation_comparison.json'
validation={name:evaluate(p,'val',name) for name,p in [
    ('baseline_no_brightness',baseline_best),
    ('experiment_brightness_035',experiment_best)]}
val_file.write_text(json.dumps(validation,indent=2))
rows=[{'run':n,'hsv_v':0.0 if 'baseline' in n else 0.35,'precision':v['precision'],
       'recall':v['recall'],'mAP50':v['mAP50'],
       'training_minutes':train_times[n]/60} for n,v in validation.items()]
comparison=pd.DataFrame(rows)
comparison.to_csv(EVIDENCE/'validation_table.csv',index=False)
display(comparison.round(4))

## 5 · Select by validation mAP50, then evaluate locked test ONCE
**Never** tune the detector after seeing test results. If mAP50 ties, choose higher validation Precision, then baseline by stable order. The chosen `best.pt` is saved as `weights/final_best.pt`.

In [ ]:
chosen=max(validation,key=lambda name:(validation[name]['mAP50'],validation[name]['precision'],
                                      name=='baseline_no_brightness'))
source_checkpoint=RUNS/chosen/'weights'/'best.pt'
final_checkpoint=WEIGHTS/'final_best.pt'
shutil.copy2(source_checkpoint,final_checkpoint)
selection={'selected_run':chosen,'criterion':'highest validation mAP50 (precision tie-break)',
   'selected_validation_metrics':validation[chosen],
   'checkpoint_sha256':hashlib.sha256(final_checkpoint.read_bytes()).hexdigest()}
(EVIDENCE/'model_selection.json').write_text(json.dumps(selection,indent=2))
test_file=EVIDENCE/'test_metrics.json'
test_report={'selected_run':chosen,'metrics':evaluate(final_checkpoint,'test','selected_model')}
test_file.write_text(json.dumps(test_report,indent=2))
print('Selected on validation:',chosen)
print('Final held-out TEST metrics:');display(pd.DataFrame([test_report['metrics']]).round(4))
print('Final checkpoint:',final_checkpoint)

## 6 · Display at least five *real* test predictions
Prediction display uses `confidence=0.25`, NMS `iou=0.7`, recorded separately from AP evaluation. Plotted labels/confidences come from model predictions; **never substitute ground-truth plots**.

In [ ]:
PRED_CONF=0.25;PRED_NMS_IOU=0.7
final_model=YOLO(str(final_checkpoint))
examples=split_files['test'][:8]
preds=final_model.predict(source=[str(p) for p in examples],imgsz=COMMON['imgsz'],
                  conf=PRED_CONF,iou=PRED_NMS_IOU,device=DEVICE,verbose=False)
fig,axs=plt.subplots(2,4,figsize=(17,9))
for ax,p,res in zip(axs.flat,examples,preds):
    im=Image.fromarray(res.plot()[:,:,::-1]);im.save(EVIDENCE/f'prediction_{p.stem}.png')
    ax.imshow(im);ax.set_title(p.stem);ax.axis('off')
plt.suptitle('REAL model predictions on locked TEST images',fontsize=16)
plt.tight_layout();plt.savefig(EVIDENCE/'test_predictions_8.png',dpi=145,bbox_inches='tight');plt.show()
(EVIDENCE/'prediction_settings.json').write_text(json.dumps({
    'conf':PRED_CONF,'nms_iou':PRED_NMS_IOU,'images':len(examples)},indent=2))

## 7 · Analyze at least three errors/challenging examples
Compare each TEST prediction against true VOC boxes. First match a prediction and ground-truth object with the **same class** and IoU >=0.5, one-to-one. Unmatched boxes are candidates for false detections, wrong classes or missed signs. Automatic error categories are **diagnostics**; inspect the generated pictures and add real visual explanations for the defense. If fewer than three failures exist, report challenging cases instead.

In [ ]:
def xyxy_gt(p):
    with Image.open(p) as im:w,h=im.size
    return [(c,((x-bw/2)*w,(y-bh/2)*h,(x+bw/2)*w,(y+bh/2)*h))
            for c,x,y,bw,bh in annotations[p.stem]]
def iou_xyxy(a,b):
    inter=max(0,min(a[2],b[2])-max(a[0],b[0]))*max(0,min(a[3],b[3])-max(a[1],b[1]))
    area=lambda q:max(0,q[2]-q[0])*max(0,q[3]-q[1])
    return inter/(area(a)+area(b)-inter+1e-9)
failures=[];challenges=[]
for st in range(0,len(split_files['test']),16):
    files=split_files['test'][st:st+16]
    rs=final_model.predict(source=[str(p) for p in files],imgsz=COMMON['imgsz'],
             conf=PRED_CONF,iou=PRED_NMS_IOU,device=DEVICE,verbose=False)
    for p,res in zip(files,rs):
        gt=xyxy_gt(p);pred=[(int(k),tuple(map(float,b)),float(conf)) for k,b,conf in zip(
            res.boxes.cls.tolist(),res.boxes.xyxy.tolist(),res.boxes.conf.tolist())]
        matched_gt=set();matched_pred=set();possible=[]
        candidates=sorted([(iou_xyxy(box,gbox),i,j) for i,(cls,box,_) in enumerate(pred)
            for j,(gcls,gbox) in enumerate(gt) if cls==gcls],reverse=True)
        for iou,i,j in candidates:
            if iou>=0.5 and i not in matched_pred and j not in matched_gt:
                matched_gt.add(j);matched_pred.add(i)
        for j,(gcls,gbox) in enumerate(gt):
            if j not in matched_gt:
                overlaps=[(iou_xyxy(gbox,box),cls) for cls,box,_ in pred]
                mx=max(overlaps,default=(0,None))
                if mx[0]>=0.5 and mx[1]!=gcls:reason='wrong class'
                elif mx[0]>=0.20:reason='inaccurate bounding box'
                else:reason='missed sign'
                possible.append(reason)
        if len(pred)>len(matched_pred):possible.append('extra / unmatched detection')
        item={'image':p.name,'failure_types':sorted(set(possible)),
              'ground_truth_objects':len(gt),'predictions':len(pred),
              'inspected_manually':False,
              'possible_causes':'Small signs, low contrast, occlusion or confusing visual patterns; verify on saved image.'}
        if possible:failures.append(item)
        elif gt:challenges.append(item)
        if len(failures)>=6:break
    if len(failures)>=6:break
review=failures[:3] if len(failures)>=3 else failures+challenges[:3-len(failures)]
for k,item in enumerate(review,1):
    p=next(p for p in split_files['test'] if p.name==item['image'])
    res=final_model.predict(source=str(p),imgsz=COMMON['imgsz'],conf=PRED_CONF,
                       iou=PRED_NMS_IOU,device=DEVICE,verbose=False)[0]
    pic=Image.open(p).convert('RGB');d=ImageDraw.Draw(pic)
    for cl,box in xyxy_gt(p):
        d.rectangle(box,outline='#0EE3B2',width=4)
        d.text((box[0],max(0,box[1]-16)),f'GT {CLASSES[cl]}',fill='#0EE3B2',stroke_width=1,stroke_fill='black')
    for cls,box,cf in zip(res.boxes.cls.tolist(),res.boxes.xyxy.tolist(),res.boxes.conf.tolist()):
        d.rectangle(tuple(map(float,box)),outline='#FFB449',width=3)
        d.text((box[0],min(pic.height-15,box[3]+3)),f'PRED {CLASSES[int(cls)]} {cf:.2f}',fill='#FFB449',stroke_width=1,stroke_fill='black')
    out=EVIDENCE/f'failure_{k:02}.png';pic.save(out)
    item['image_file']=out.name
(EVIDENCE/'failure_analysis_candidates.json').write_text(json.dumps({
    'actual_automatic_failures_found':len(failures),
    'diagnostic_IoU_match_threshold':0.5,
    'annotation':'GREEN = ground truth; ORANGE = predictions; inspect for real causes',
    'cases':review},indent=2))
print('Failure candidates:',len(failures),'| Saved representative cases:',len(review))
for row in review:print(row['image'],':',row['failure_types'] or ['challenging but correct'])

## 8 · Export real evidence and the selected weights
Run this cell only after both training runs and evaluation finish. Download the generated ZIP and publish real evidence plus an **accessible checkpoint link** on GitHub or external storage. Colab notebook itself can be pushed to GitHub; do not commit the dataset or GPU run files directly unless rights and size allow it.

In [ ]:
for name in ['baseline_no_brightness','experiment_brightness_035']:
    r=RUNS/name
    for filename in ['results.csv','results.png','args.yaml']:
        if (r/filename).exists():shutil.copy2(r/filename,EVIDENCE/f'{name}_{filename}')
    if (r/'weights'/'best.pt').exists():
        shutil.copy2(r/'weights'/'best.pt',WEIGHTS/f'{name}_best.pt')
summary={'source':DATA_URL,'source_kaggle':'https://www.kaggle.com/datasets/andrewmvd/road-sign-detection',
 'classes':CLASSES,'data':counts,'split_seed':SEED,'split_method':'VOC provider holdout and near-duplicate groups',
 'environment':ENV,'fixed_configuration':COMMON,'experiment':'hsv_v: 0.0 vs 0.35',
 'hypothesis':'Brightness jitter may increase validation mAP50',
 'train_times_sec':train_times,'validation':validation,'selection':selection,
 'test':test_report,'evaluation_thresholds':{'conf':EVAL_CONF,'NMS_IoU':EVAL_NMS_IOU,'AP_IoU':0.5},
 'prediction_thresholds':{'conf':PRED_CONF,'NMS_IoU':PRED_NMS_IOU},
 'caveat':'No original video IDs; visually inspect near-duplicate image grouping'}
(EVIDENCE/'results_summary.json').write_text(json.dumps(summary,indent=2))
import zipfile
bundle=BASE/'TRAFFIC_SIGN_SUBMISSION_ARTIFACTS.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED,compresslevel=5) as z:
    for folder in [EVIDENCE,WEIGHTS,RUNS]:
        for p in folder.rglob('*'):
            if p.is_file():z.write(p,p.relative_to(BASE))
    z.write(DATA_YAML,'traffic_signs.yaml')
print('ARTIFACTS BUNDLE:',bundle)
print('FINAL WEIGHTS:',final_checkpoint)
print('Selected by validation:',chosen)
print('Final TEST Precision/Recall/mAP50:',
      round(test_report['metrics']['precision'],4),round(test_report['metrics']['recall'],4),
      round(test_report['metrics']['mAP50'],4))
try:
    from google.colab import files as colab_files
    colab_files.download(str(bundle))
except ImportError:
    print('Not in Colab; download this ZIP manually:',bundle)